# SentinelID - Phase 1: Tampering Detection (P2)
Run each cell in order. Upload a test document image when prompted.

In [ ]:
!pip install opencv-python-headless pillow numpy scikit-image -q

## Step 1: Upload test image

In [ ]:
from google.colab import files
uploaded = files.upload()
image_path = list(uploaded.keys())[0]

## Step 2: Error Level Analysis (ELA)
Detects regions re-saved/edited at a different JPEG compression level than the rest of the image.

In [ ]:
from PIL import Image, ImageChops, ImageEnhance

def compute_ela(image_path, quality=90):
    original = Image.open(image_path).convert('RGB')
    original.save("temp_resaved.jpg", quality=quality)
    resaved = Image.open("temp_resaved.jpg")
    diff = ImageChops.difference(original, resaved)
    extrema = diff.getextrema()
    max_diff = max([ex[1] for ex in extrema])
    scale = 255.0 / max_diff if max_diff != 0 else 1
    ela_image = ImageEnhance.Brightness(diff).enhance(scale)
    ela_image.save("ela_output.jpg")
    return ela_image, max_diff

ela_img, max_diff = compute_ela(image_path)
ela_img  # displays inline in Colab

## Step 3: Metadata/EXIF check

In [ ]:
from PIL import Image

def check_metadata(image_path):
    img = Image.open(image_path)
    exif = img._getexif()
    flags = []
    if exif is None:
        flags.append("No EXIF data - possible screenshot/edited export")
    return flags

metadata_flags = check_metadata(image_path)
metadata_flags

## Step 4: Combined tamper scoring function
This is the function to export to your teammates for Phase 2 backend wiring.

In [ ]:
def tamper_score(image_path):
    ela_img, max_diff = compute_ela(image_path)
    metadata_flags = check_metadata(image_path)
    score = min(max_diff / 100 * 100, 100)
    verdict = "SUSPICIOUS" if score > 40 or metadata_flags else "CLEAN"
    return {
        "tamper_score": round(score, 2),
        "metadata_flags": metadata_flags,
        "verdict": verdict
    }

import json
print(json.dumps(tamper_score(image_path), indent=2))

## Step 5 (stretch): Quick MobileNet tamper classifier scaffold
Only build this out if time allows in Phase 4 - train on MIDV-500 genuine/tampered crops.

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(1, activation='sigmoid')(x)  # 0 = genuine, 1 = tampered
model = Model(inputs=base_model.input, outputs=predictions)

for layer in base_model.layers:
    layer.trainable = False

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()
# Train on MIDV-500 genuine/tampered crops if time allows (Phase 4 stretch)

## Exit test
Genuine sample gives CLEAN with a low score. Manually edit one image in any photo editor, re-upload, and it should give SUSPICIOUS with a high score.

Download this notebook as .py via File > Download > Download .py once done.